# RuleShift Benchmark

## Task overview

This benchmark evaluates RuleShift as a cognitive ability task focused on how well a model adapts its behavior when the governing rule changes. The goal is not simple recall, but consistent response adjustment under shifting constraints.
This notebook defines the full benchmark flow used for evaluation, including dataset loading, response normalization, per-episode scoring, task registration, and final task selection for Kaggle Benchmarks.

## Dataset structure

Each dataset is a flat JSON array of episode objects with the schema `{"episode_id": "0001", "inference": {"prompt": "..."}, "scoring": {"probe_targets": [...]}, "analysis": {"group_id": "simple|exception|distractor|hard", "rule_id": "...", "shortcut_type": "...", "shortcut_rule_id": "...|null"}}`.
The packaged public split contains 80 audited episodes total with 20 episodes per group, and the packaged private split contains 400 audited episodes total with 100 episodes per group.

| What this notebook does | Why it matters |
| --- | --- |
| Loads the frozen benchmark rows | Ensures every submission is evaluated on the same prompts |
| Normalizes model outputs | Makes scoring robust to a few response formats |
| Scores every episode | Produces the final leaderboard numerator and denominator |
| Registers the Kaggle task | Exposes the benchmark through the expected entry point |

---

> Recommended flow: run the notebook top to bottom once, confirm the dataset loads correctly, then use the registered task as the official benchmark entry.

## Notebook Map

<div style="background:#111;border:1px solid #2a2a2a;border-radius:6px;padding:18px 22px;font-family:monospace;margin-bottom:8px">
  <div style="display:flex;align-items:center;flex-wrap:wrap;gap:4px">
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Paths</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Types</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Normalize</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Score</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Load Rows</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Register Task</span>
    <span style="color:#444;font-size:18px">╎</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Smoke Check</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#ccc;font-size:12px">Choose Task</span>
  </div>
</div>

## Phase 1 — Runtime setup

Defines the constants, types, and filesystem paths that every later cell depends on.

### Step 1 — Locate the packaged benchmark data

This cell defines the runtime dataset paths before any benchmark logic starts.

- **`RULESHIFT_EVAL_SPLIT`** — selects the frozen split to load; defaults to `"public"`
- **`DEFAULT_DATASET_ROOT`** — canonical public Kaggle path; used unless overridden
- **`RULESHIFT_DATASET_ROOT`** — set this env var to point at a different public dataset root (e.g. locally)
- **`RULESHIFT_PRIVATE_DATASET_ROOT`** — optional override for the private dataset root

The cell fails immediately if the selected split cannot be resolved, so path errors surface at the top rather than mid-run.

In [ ]:
import os
from pathlib import Path

import kaggle_benchmarks as kbench
import pandas as pd

DATASET_ROOT_ENV_VAR = "RULESHIFT_DATASET_ROOT"
PRIVATE_DATASET_ROOT_ENV_VAR = "RULESHIFT_PRIVATE_DATASET_ROOT"
EVAL_SPLIT = os.environ.get("RULESHIFT_EVAL_SPLIT", "public").strip().lower()
EXPECTED_PUBLIC_EPISODE_COUNT = 80
EXPECTED_PRIVATE_EPISODE_COUNT = 400
EXPECTED_EPISODES_PER_GROUP = {"public": 20, "private": 100}
SMOKE_CHECK_EPISODE_COUNT = 3

DEFAULT_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime")
DEFAULT_PRIVATE_DATASET_ROOT = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime-private")
DATASET_ROOT = Path(os.environ.get(DATASET_ROOT_ENV_VAR, DEFAULT_DATASET_ROOT))
PRIVATE_DATASET_ROOT = Path(
    os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR, DEFAULT_PRIVATE_DATASET_ROOT)
)

if EVAL_SPLIT not in {"public", "private"}:
    raise ValueError(f"Unsupported EVAL_SPLIT: {EVAL_SPLIT!r}")

if EVAL_SPLIT == "public":
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"Public dataset root not found: {DATASET_ROOT}\n"
            f"Set {DATASET_ROOT_ENV_VAR} to override the default location."
        )
    ROWS_PATH = DATASET_ROOT / "public_leaderboard_rows.json"
else:
    if not PRIVATE_DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"Private split selected but private dataset root not found: {PRIVATE_DATASET_ROOT}\n"
            f"Set {PRIVATE_DATASET_ROOT_ENV_VAR} to override the private dataset location."
        )
    ROWS_PATH = PRIVATE_DATASET_ROOT / "private_leaderboard_rows.json"

if not ROWS_PATH.exists():
    raise FileNotFoundError(
        f"Selected split rows file not found: {ROWS_PATH}"
    )

### Step 2 — Define shared constants and types

The runtime is intentionally split into smaller sections so each concern can be understood on its own.

| Runtime section | What it contains |
| --- | --- |
| Shared constants and types | Labels, response schema, path config, and benchmark-wide settings |
| Response normalization | Logic that converts different model outputs into the expected labels |
| Scoring helpers | Functions that compare predictions against the frozen targets |
| Dataset loading | Functions that read the public and private benchmark rows |

This first code block defines the shared vocabulary that every later cell depends on.

In [ ]:
import re
from collections import Counter
from dataclasses import dataclass
from enum import Enum
from typing import Final

class Label(str, Enum):
    type_a = "type_a"
    type_b = "type_b"

EXAMPLE_COUNT: Final[int] = 5
PROBE_COUNT: Final[int] = 4
PROMPT_HEADER_PREFIX: Final[str] = "RuleShift classification task. Episode "
EXAMPLES_HEADER: Final[str] = "Examples:\n"
PROBES_HEADER: Final[str] = "Probes:\n"
OUTPUT_INSTRUCTION: Final[str] = (
    "Return exactly 4 outputs in order, one per probe. "
    "Use only type_a or type_b. Map zark to type_a and blim to type_b."
)
ALLOWED_GROUPS: Final[frozenset[str]] = frozenset({"simple", "exception", "distractor", "hard"})
EXCLUDED_GROUPS: Final[frozenset[str]] = frozenset({"multi-step"})
FORBIDDEN_PROMPT_SNIPPETS: Final[tuple[str, ...]] = (
    "multi-step",
    "step 1:",
    "step 2:",
    "step 3:",
    "intentionally direct",
    "broad pattern",
    "authoritative",
    "simpler trend",
    "ignore the audit tags",
    "bookkeeping noise",
    "easy shortcut",
    "competing local patterns",
    "exception-setting row",
    "previous turn",
    "next turn",
)
BENCHMARK_INVARIANTS: Final[tuple[str, ...]] = (
    "group_id must stay within the scoped single-turn groups",
    "each prompt must contain exactly five labeled examples and four probes",
    "the prompt frame must stay fixed across the scoped groups",
    "the answer format must stay as four ordered type_a/type_b labels",
    "probe_targets must stay aligned to the four displayed probes",
    "prompts must not introduce group-specific coaching, multi-step scaffolding, or multi-turn language",
)

_ALLOWED_LABELS: Final[frozenset[str]] = frozenset(label.value for label in Label)
_PROBE_FIELDS: Final[tuple[str, ...]] = ("probe_6", "probe_7", "probe_8", "probe_9")

@dataclass(frozen=True)
class BinaryResponse:
    probe_6: Label
    probe_7: Label
    probe_8: Label
    probe_9: Label

    def as_tuple(self) -> tuple[str, str, str, str]:
        """Returns the four probe labels as normalized strings.

        Converts each stored probe label into its canonical lowercase string.

        Returns:
            A four-item tuple containing the normalized probe labels.
        """
        return (
            _coerce_label(self.probe_6, "probe_6"),
            _coerce_label(self.probe_7, "probe_7"),
            _coerce_label(self.probe_8, "probe_8"),
            _coerce_label(self.probe_9, "probe_9"),
        )

class KaggleExecutionError(RuntimeError):
    pass

## Phase 2 — Scoring logic

Implements response normalization and the episode-level scoring functions.

### Step 3 — Normalize model responses

Model outputs are not always shaped the same way, so this section standardizes them before scoring.

| Accepted shape | Example |
| --- | --- |
| Structured response object | `BinaryResponse(...)` |
| Dictionary-like response | `{"probe_6": "type_a", ...}` |
| Plain text response | `type_a, type_b, type_a, type_b` |

The helpers are forgiving about format, but strict about labels: each normalized answer must end up as exactly four values drawn from `type_a` and `type_b`.

In [ ]:
def normalize_binary_response(response: object) -> tuple[str, ...] | None:
    """Normalizes a model response into a canonical four-label tuple.

    Accepts structured objects, mapping-like objects, or plain text responses.

    Args:
        response: The raw model response to normalize.

    Returns:
        A normalized four-label tuple, or `None` when the response cannot be normalized.
    """
    if response is None:
        return None
    if isinstance(response, BinaryResponse):
        return response.as_tuple()
    if isinstance(response, str):
        return _parse_text_response(response)
    br = _try_coerce(response)
    return br.as_tuple() if br is not None else None

def _parse_text_response(text: str) -> tuple[str, ...] | None:
    """Parses a plain-text response into normalized probe labels.

    Splits text on commas after trimming whitespace, backticks, and line breaks.

    Args:
        text: The plain-text response to parse.

    Returns:
        A normalized four-label tuple, or `None` if parsing does not yield four valid labels.
    """
    tokens = tuple(
        t.strip().lower()
        for t in text.strip().strip("`").replace("\n", ",").split(",")
        if t.strip()
    )
    return _norm_labels(tokens)

def _try_coerce(response: object) -> BinaryResponse | None:
    """Attempts to coerce a response object into `BinaryResponse`.

    Supports dictionaries, mapping-like objects, and objects exposing probe attributes.

    Args:
        response: The response object to coerce.

    Returns:
        A `BinaryResponse` instance, or `None` when coercion fails.
    """
    if isinstance(response, dict):
        vals = response
    elif hasattr(response, "__getitem__") and hasattr(response, "keys"):
        try:
            vals = {f: response[f] for f in _PROBE_FIELDS}
        except (KeyError, TypeError):
            return None
    elif all(hasattr(response, f) for f in _PROBE_FIELDS):
        vals = {f: getattr(response, f) for f in _PROBE_FIELDS}
    else:
        return None
    try:
        labels = tuple(Label(_coerce_label(vals[f], f)) for f in _PROBE_FIELDS)
    except (KeyError, TypeError):
        return None
    return BinaryResponse(*labels)

def _coerce_label(value: object, field: str) -> str:
    """Normalizes a single label value with field-aware errors.

    Wraps label normalization so invalid values report which field failed.

    Args:
        value: The label value to normalize.
        field: The field name associated with the value.

    Returns:
        The normalized label string.
    """
    try:
        return _normalize_label(value)
    except ValueError as exc:
        raise ValueError(f"invalid field {field}: {value!r}") from exc

def _norm_labels(
    labels: tuple[str, ...] | tuple[Label, ...] | None,
) -> tuple[str, ...] | None:
    """Normalizes a sequence of labels and enforces the probe count.

    Validates that the sequence contains exactly four supported labels.

    Args:
        labels: The label sequence to normalize.

    Returns:
        A normalized four-label tuple, or `None` if validation fails.
    """
    if labels is None:
        return None
    try:
        result = tuple(
            _normalize_label(lbl.value if isinstance(lbl, Label) else lbl)
            for lbl in labels
        )
    except ValueError:
        return None
    return result if len(result) == PROBE_COUNT else None

def _normalize_label(value: object) -> str:
    """Converts a supported label value to its canonical string form.

    Accepts enum values and other value-like objects with a `.value` attribute.

    Args:
        value: The value to normalize.

    Returns:
        The lowercase canonical label string.
    """
    if isinstance(value, Enum):
        value = value.value
    elif hasattr(value, "value"):
        value = value.value
    normalized = str(value).strip().lower()
    if normalized not in _ALLOWED_LABELS:
        raise ValueError(f"unknown label: {value}")
    return normalized

### Step 4 — Score each benchmark episode

This section converts one benchmark prompt into one measurable score.

<div style="background:#111;border:1px solid #2a2a2a;border-radius:6px;padding:18px 22px;font-family:monospace;margin-bottom:8px">
  <div style="display:flex;align-items:center;flex-wrap:wrap;gap:4px">
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Prompt the model</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Normalize the response</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#aaa;font-size:12px">Compare with targets</span>
    <span style="color:#333;font-size:18px">→</span>
    <span style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:4px;padding:4px 10px;color:#ccc;font-size:12px">Return (numerator, denominator)</span>
  </div>
</div>

If the model call fails or returns something unscoreable, the runtime raises an explicit error so the failure is easy to diagnose.

In [ ]:
def score_episode(
    predictions: tuple[str, ...] | tuple[Label, ...] | None,
    targets: tuple[str, ...] | tuple[Label, ...],
) -> tuple[int, int]:
    """Scores one episode by comparing predictions with target labels.

    Treats invalid or missing predictions as zero correct answers.

    Args:
        predictions: The predicted probe labels.
        targets: The target probe labels.

    Returns:
        A `(numerator, denominator)` tuple for the episode score.
    """
    norm_targets = _norm_labels(targets)
    if norm_targets is None:
        raise ValueError(f"targets must contain exactly {PROBE_COUNT} valid labels")
    norm_preds = _norm_labels(predictions)
    if norm_preds is None:
        return (0, PROBE_COUNT)
    return (
        sum(p == t for p, t in zip(norm_preds, norm_targets)),
        PROBE_COUNT,
    )

@kbench.task(store_task=False)
def run_binary_task(
    llm: object,
    prompt: str,
    probe_targets: tuple[str, ...] | tuple[Label, ...],
) -> tuple[int, int]:
    """Runs one benchmark episode against the provided model.

    Prompts the model, normalizes its response, and scores the resulting labels.

    Args:
        llm: The model interface used to produce a response.
        prompt: The benchmark prompt to send to the model.
        probe_targets: The expected probe labels for scoring.

    Returns:
        A `(numerator, denominator)` tuple for the episode score.
    """
    try:
        response = llm.prompt(prompt, schema=BinaryResponse)
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed") from exc

    try:
        normalized = normalize_binary_response(response)
    except ValueError as exc:
        raise KaggleExecutionError(f"invalid binary response: {exc}") from exc

    if normalized is None:
        raise KaggleExecutionError(
            f"unscoreable response of type {type(response).__name__}"
        )
    return score_episode(normalized, probe_targets)

## Phase 3 — Dataset loading

Materializes the frozen evaluation rows from the packaged runtime dataset.

### Step 5 — Load the frozen benchmark rows

This part materializes the full evaluation set that the task will evaluate.

| Row set | Source | When it is used |
| --- | --- | --- |
| Public rows | Packaged public dataset | When `EVAL_SPLIT == "public"` |
| Private rows | Packaged private dataset or `RULESHIFT_PRIVATE_DATASET_ROOT` | When `EVAL_SPLIT == "private"` |

At the end of the cell, `df` becomes the single source of truth for the benchmark batch evaluation flow. If no rows are available, the notebook stops immediately.

In [ ]:
import json

def load_selected_rows() -> list[dict[str, object]]:
    """Loads the leaderboard rows for the selected evaluation split.

    Reads the benchmark rows for the currently selected split.

    Returns:
        A list of normalized benchmark rows.
    """
    return _load_rows(ROWS_PATH)

def _count_prompt_rows(prompt: str) -> tuple[int, int]:
    """Counts labeled example rows and unlabeled probe rows inside a prompt."""
    parts = prompt.split("\n\n")
    if len(parts) != 4:
        return (0, 0)

    _, examples_block, probes_block, _ = parts
    example_lines = tuple(line.strip() for line in examples_block.splitlines()[1:] if line.strip())
    probe_lines = tuple(line.strip() for line in probes_block.splitlines()[1:] if line.strip())
    example_count = sum(_matches_prompt_row(line, ("zark", "blim")) for line in example_lines)
    probe_count = sum(_matches_prompt_row(line, ("?",)) for line in probe_lines)
    return (example_count, probe_count)

def _matches_prompt_row(line: str, expected_labels: tuple[str, ...]) -> bool:
    """Checks whether a prompt line matches the frozen row format."""
    row_prefix = r"(?:\d+\.|\[\d+\]|case \d+|item \d+)"
    label_slot = r"(?:->|observed=|state=|mark=)"
    labels = "|".join(re.escape(label) for label in expected_labels)
    pattern = rf"^{row_prefix}\s*.+?\s*{label_slot}\s*(?:{labels})\s*$"
    return re.fullmatch(pattern, line) is not None

def _validate_prompt_contract(prompt: str, episode_id: str) -> None:
    """Validates the neutral single-turn prompt frame used by the benchmark."""
    parts = prompt.split("\n\n")
    if len(parts) != 4:
        raise ValueError(
            f"episode {episode_id}: expected 4 prompt blocks, found {len(parts)}"
        )

    header, examples_block, probes_block, output_block = parts
    if not header.startswith(PROMPT_HEADER_PREFIX):
        raise ValueError(
            f"episode {episode_id}: prompt must start with the benchmark task header"
        )
    if not examples_block.startswith(EXAMPLES_HEADER):
        raise ValueError(
            f"episode {episode_id}: prompt must use the standard Examples section"
        )
    if not probes_block.startswith(PROBES_HEADER):
        raise ValueError(
            f"episode {episode_id}: prompt must use the standard Probes section"
        )
    if output_block != OUTPUT_INSTRUCTION:
        raise ValueError(
            f"episode {episode_id}: prompt must use the fixed benchmark output instruction"
        )

def _validate_row(row: dict[str, object]) -> None:
    """Validates that a row stays inside the frozen single-turn scope."""
    episode_id = row["episode_id"]
    analysis = row["analysis"]
    group_id = analysis["group_id"]
    prompt = row["inference"]["prompt"]

    if group_id not in ALLOWED_GROUPS:
        raise ValueError(f"episode {episode_id}: unsupported benchmark group_id {group_id!r}")
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError(f"episode {episode_id}: prompt must be a non-empty string")
    _validate_prompt_contract(prompt, str(episode_id))

    lower_prompt = prompt.lower()
    forbidden = [token for token in FORBIDDEN_PROMPT_SNIPPETS if token in lower_prompt]
    if forbidden:
        raise ValueError(
            f"episode {episode_id}: prompt violates benchmark scope via {forbidden}"
        )

    example_count, probe_count = _count_prompt_rows(prompt)
    if example_count != EXAMPLE_COUNT:
        raise ValueError(
            f"episode {episode_id}: expected {EXAMPLE_COUNT} labeled examples, found {example_count}"
        )
    if probe_count != PROBE_COUNT:
        raise ValueError(
            f"episode {episode_id}: expected {PROBE_COUNT} probes, found {probe_count}"
        )

def _validate_batch(rows: list[dict[str, object]]) -> None:
    """Validates scoped row counts for the selected split."""
    expected_total = (
        EXPECTED_PUBLIC_EPISODE_COUNT
        if EVAL_SPLIT == "public"
        else EXPECTED_PRIVATE_EPISODE_COUNT
    )
    if len(rows) != expected_total:
        raise RuntimeError(
            f"Scoped benchmark dataset mismatch: expected {expected_total} episodes "
            f"for split {EVAL_SPLIT!r}, but loaded {len(rows)} from {ROWS_PATH}."
        )

    counts = Counter(str(row["analysis"]["group_id"]) for row in rows)
    rule_counts = Counter(str(row["analysis"]["rule_id"]) for row in rows)
    shortcut_counts = Counter(str(row["analysis"]["shortcut_type"]) for row in rows)
    label_counts = Counter(
        label
        for row in rows
        for label in row["scoring"]["probe_targets"]
    )
    expected_per_group = EXPECTED_EPISODES_PER_GROUP[EVAL_SPLIT]
    if set(counts) != ALLOWED_GROUPS:
        raise RuntimeError(
            f"Scoped benchmark groups mismatch: expected {sorted(ALLOWED_GROUPS)}, "
            f"got {sorted(counts)}"
        )
    bad_counts = {
        group_id: count
        for group_id, count in counts.items()
        if count != expected_per_group
    }
    if bad_counts:
        raise RuntimeError(
            f"Scoped benchmark per-group counts mismatch for split {EVAL_SPLIT!r}: {bad_counts}"
        )
    print(f"Scoped group counts: {dict(sorted(counts.items()))}")
    print(f"Scoped rule count: {len(rule_counts)}")
    print(f"Scoped shortcut counts: {dict(sorted(shortcut_counts.items()))}")
    print(f"Scoped label counts: {dict(sorted(label_counts.items()))}")

def _load_rows(path: Path) -> list[dict[str, object]]:
    """Loads benchmark rows from disk and normalizes their probe targets.

    Extracts the fields required by the runtime from the raw JSON payload.

    Args:
        path: The path to the JSON file containing benchmark rows.

    Returns:
        A list of normalized benchmark row dictionaries.
    """
    rows = json.loads(path.read_text("utf-8"))
    scoped_rows: list[dict[str, object]] = []
    excluded_episode_ids: list[str] = []
    unexpected_groups: set[str] = set()

    for row in rows:
        analysis = row.get("analysis")
        inference = row.get("inference")
        scoring = row.get("scoring")
        if not isinstance(analysis, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing analysis metadata")
        if not isinstance(inference, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing inference fields")
        if not isinstance(scoring, dict):
            raise RuntimeError(f"row {row.get('episode_id')} is missing scoring fields")

        group_id = analysis["group_id"]
        if group_id in EXCLUDED_GROUPS:
            excluded_episode_ids.append(str(row["episode_id"]))
            continue
        if group_id not in ALLOWED_GROUPS:
            unexpected_groups.add(str(group_id))
            continue

        normalized_row = {
            "episode_id": row["episode_id"],
            "inference": {
                "prompt": inference["prompt"],
            },
            "scoring": {
                "probe_targets": _normalize_probe_targets(scoring["probe_targets"]),
            },
            "analysis": {
                "group_id": group_id,
                "rule_id": analysis["rule_id"],
                "shortcut_type": analysis["shortcut_type"],
                "shortcut_rule_id": analysis.get("shortcut_rule_id"),
            },
        }
        _validate_row(normalized_row)
        scoped_rows.append(normalized_row)

    if unexpected_groups:
        raise RuntimeError(
            f"Found unsupported benchmark groups in {path}: {sorted(unexpected_groups)}"
        )
    if excluded_episode_ids:
        print(
            f"Excluded {len(excluded_episode_ids)} legacy out-of-scope episodes "
            f"from groups {sorted(EXCLUDED_GROUPS)}."
        )

    _validate_batch(scoped_rows)
    return scoped_rows

def _normalize_probe_targets(values: object) -> tuple[str, ...]:
    """Validates and normalizes stored probe target labels.

    Ensures the targets are provided as a sequence of exactly four valid labels.

    Args:
        values: The raw probe target values to validate.

    Returns:
        A normalized four-label tuple.
    """
    if not isinstance(values, list | tuple):
        raise ValueError("probe_targets must be a list or tuple")
    labels = _norm_labels(tuple(values))
    if labels is None:
        raise ValueError(f"probe_targets must contain exactly {PROBE_COUNT} valid labels")
    return labels

leaderboard_rows = load_selected_rows()
loaded_episode_count = len(leaderboard_rows)
print(f"Dataset rows path: {ROWS_PATH}")
print(f"Selected split: {EVAL_SPLIT}")
print(f"Loaded episodes: {loaded_episode_count}")

if not leaderboard_rows:
    raise RuntimeError("No benchmark episodes were loaded.")

df = pd.DataFrame(
    [
        {
            "prompt": row["inference"]["prompt"],
            "probe_targets": row["scoring"]["probe_targets"],
        }
        for row in leaderboard_rows
    ]
)

## Phase 4 — Task registration

Wraps the scoring loop into the `@kbench.task` entry point exposed to Kaggle.

### Step 6 — Register the official Kaggle task

This cell turns the batch evaluation flow into a Kaggle Benchmarks task.

Execution sequence:

1. Evaluate `run_binary_task` once per row in `df`.
2. Send each prompt to the provided `llm`.
3. Score the four probe predictions for that episode.
4. Aggregate the returned episode totals into a final benchmark result.
5. Save a compact summary so the notebook can display it after the run.

This is the main evaluation cell: everything above prepares data and helpers for this flow.

In [ ]:
def summarize_ruleshift_benchmark(runs, episode_count: int) -> dict[str, int | float]:
    """Builds the final benchmark summary from an evaluation run."""
    if not runs:
        raise RuntimeError("evaluation produced no successful runs")

    results_df = runs.as_dataframe()
    if len(results_df) != episode_count:
        raise RuntimeError(
            f"evaluation completed {len(results_df)} of {episode_count} benchmark episodes"
        )

    numerator = int(results_df.result.str.get(0).sum())
    denominator = int(results_df.result.str.get(1).sum())

    if denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")

    return {
        "score": numerator / denominator,
        "numerator": numerator,
        "denominator": denominator,
        "episodes": episode_count,
    }


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Single-turn cognitive flexibility benchmark: infer the active "
        "within-prompt labeling rule from five examples, then predict "
        "four ordered probe outputs without multi-step or multi-turn behavior."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> tuple[int, int]:
    """Evaluates the benchmark across all loaded rows.

    Aggregates episode-level scores for the full benchmark run.

    Args:
        llm: The Kaggle Benchmarks-compatible model interface to evaluate.

    Returns:
        A `(numerator, denominator)` tuple for the full benchmark score.
    """
    summary = summarize_ruleshift_benchmark(
        run_binary_task.evaluate(llm=[llm], evaluation_data=df),
        len(df),
    )

    print(json.dumps(summary, indent=2))
    return (summary["numerator"], summary["denominator"])

## Phase 5 — Diagnostics and selection

Keeps optional diagnostics outside the official score and binds the entry point for Kaggle.

### Step 7 — Define optional diagnostics helpers

These helpers stay outside the official benchmark score. Use them after an evaluation run when you want a quick breakdown by internal analysis fields.

- **By group** — compare `simple`, `exception`, `distractor`, and `hard`
- **By internal category** — summarize by `rule_id` and `shortcut_type`
- **By miss pattern** — count episodes with 0, 1, 2, 3, or 4 correct probes

> **Usage:** run the official task, keep the `runs` object from `run_binary_task.evaluate(...)`, then call `build_ruleshift_diagnostics(runs, leaderboard_rows)`.

In [ ]:
def build_ruleshift_diagnostics(runs, rows: list[dict[str, object]]) -> dict[str, object]:
    """Builds optional post-run diagnostics without changing the official score."""
    results_df = runs.as_dataframe().reset_index(drop=True)
    if len(results_df) != len(rows):
        raise RuntimeError(
            f"diagnostics expected {len(rows)} runs, but received {len(results_df)}"
        )

    diagnostics_df = pd.DataFrame(
        {
            "episode_id": [row["episode_id"] for row in rows],
            "group_id": [row["analysis"]["group_id"] for row in rows],
            "rule_id": [row["analysis"]["rule_id"] for row in rows],
            "shortcut_type": [row["analysis"]["shortcut_type"] for row in rows],
            "shortcut_rule_id": [row["analysis"]["shortcut_rule_id"] for row in rows],
            "numerator": [int(result[0]) for result in results_df.result],
            "denominator": [int(result[1]) for result in results_df.result],
        }
    )
    diagnostics_df["accuracy"] = diagnostics_df["numerator"] / diagnostics_df["denominator"]
    diagnostics_df["miss_count"] = diagnostics_df["denominator"] - diagnostics_df["numerator"]

    def _aggregate(column: str) -> list[dict[str, object]]:
        grouped = (
            diagnostics_df.groupby(column, dropna=False)
            .agg(
                episodes=("episode_id", "count"),
                numerator=("numerator", "sum"),
                denominator=("denominator", "sum"),
            )
            .reset_index()
        )
        grouped["accuracy"] = grouped["numerator"] / grouped["denominator"]
        return grouped.sort_values(["accuracy", column], ascending=[False, True]).to_dict("records")

    miss_pattern = (
        diagnostics_df.groupby("miss_count")
        .agg(episodes=("episode_id", "count"))
        .reset_index()
        .sort_values("miss_count")
        .to_dict("records")
    )

    return {
        "overall": {
            "episodes": int(diagnostics_df.shape[0]),
            "numerator": int(diagnostics_df["numerator"].sum()),
            "denominator": int(diagnostics_df["denominator"].sum()),
            "accuracy": float(diagnostics_df["numerator"].sum() / diagnostics_df["denominator"].sum()),
        },
        "by_group": _aggregate("group_id"),
        "by_rule": _aggregate("rule_id"),
        "by_shortcut_type": _aggregate("shortcut_type"),
        "miss_pattern": miss_pattern,
    }

### Step 8 — Mark the official entry point

The final step is to tell Kaggle which registered task name should be used as the notebook entry point.

In [ ]:
%choose ruleshift_benchmark_v1_binary